# Probar modelo - Clasificador de Animales

Carga el modelo entrenado y permite predecir imágenes.

In [2]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

import sys
import numpy as np
from PIL import Image
from io import BytesIO
import tensorflow as tf
import ipywidgets as widgets
from IPython.display import display, clear_output

sys.path.insert(0, os.getcwd())
from procesar_imagen import remover_fondo

TAMANO = 64
CLASES = ["aranas", "ballenas", "changos", "pajaros", "ranas"]

modelo = tf.keras.models.load_model('modelo_animales.keras')
print("Modelo cargado.")

E0000 00:00:1779901905.738069    5703 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
I0000 00:00:1779901905.738370    5703 cuda_diagnostics.cc:160] env: CUDA_VISIBLE_DEVICES="-1"
I0000 00:00:1779901905.738722    5703 cuda_diagnostics.cc:163] CUDA_VISIBLE_DEVICES is set to -1 - this hides all GPUs from CUDA
I0000 00:00:1779901905.738735    5703 cuda_diagnostics.cc:171] verbose logging is disabled. Rerun with verbose logging (usually --v=1 or --vmodule=cuda_diagnostics=1) to get more diagnostic output from this module
I0000 00:00:1779901905.738738    5703 cuda_diagnostics.cc:176] retrieving CUDA diagnostic information for host: pop-os
I0000 00:00:1779901905.738744    5703 cuda_diagnostics.cc:183] hostname: pop-os
I0000 00:00:1779901905.739740    5703 cuda_diagnostics.cc:190] libcuda reported version is: 580.159.3
I0000 00:00:1779901905.739796    5703 cuda_diagnostics.cc:194] kernel reported

Modelo cargado.


In [3]:
uploader = widgets.FileUpload(
    accept='image/*', multiple=False,
    description='Seleccionar imagen',
    style={'button_color': '#4CAF50'}
)
output = widgets.Output()

def predecir_imagen(change):
    with output:
        clear_output()
        archivos = change['new']
        if not archivos:
            return
        for info in archivos:
            if isinstance(info, dict):
                nombre = info.get('name', 'imagen')
                content = info.get('content', None)
                if content is None:
                    continue
            else:
                nombre = 'imagen'
                content = info
            img_pil = Image.open(BytesIO(content)).convert('RGB')
            img_resized = img_pil.resize((TAMANO, TAMANO))
            img_sf = remover_fondo(img_resized)

            def _pred(x): return modelo.predict(x, verbose=0)[0]
            pred_orig = _pred(np.expand_dims(np.array(img_resized, dtype=np.float32) / 255.0, axis=0))
            pred_sf = _pred(np.expand_dims(np.array(img_sf, dtype=np.float32) / 255.0, axis=0))
            pred = pred_sf if pred_sf.max() >= pred_orig.max() else pred_orig

            idx = np.argmax(pred)
            print(f"Archivo: {nombre}")
            print(f"Prediccion: {CLASES[idx]} ({pred[idx]:.1%})")
            print()
            print("Probabilidades:")
            for i, clase in enumerate(CLASES):
                barra = '█' * int(pred[i] * 40)
                print(f"  {clase:10s} {pred[i]:7.1%} {barra}")
            print()
            display(img_pil.resize((200, 200)))

uploader.observe(predecir_imagen, names='value')
display(uploader, output)

FileUpload(value=(), accept='image/*', description='Seleccionar imagen', style=ButtonStyle(button_color='#4CAF…

Output()